# 7-to-1 |Y⟩ State Distillation (Steane Code)

Reference: `processing/Steane_distillation.png` Figure 46(c)

**Circuit structure:**
- W1, W2, W3: injected |Y⟩ (noisy)
- W4: fault-tolerant |+⟩
- A: ancilla |Y⟩ (reused per π/4 rotation)
- 4 sequential Z product measurements
- End: MX on W1-W3 (post-select), S†+MX on W4 (output)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.qec_code.surface_code.unrotated import (
    UnrotatedSurfaceCode,
    UnrotatedSurfaceCodeExtractionBlock,
    UnrotatedMultiPatchCoupler,
    UnrotatedSurfaceCodeLogicalOpSet,
)
from src.ir.qec_system import QECSystem
from src.ir.builder import CircuitBuilder
from src.ir.tracker import SyndromeTracker
import numpy as np

## Step 1: Single Z Product Measurement (copy of working Exp 2 pattern)

Exact same pattern as `test_unrotated_multi_coupler.ipynb` Exp 2.
3 patches: ZZZ measurement with detectors.

In [ ]:
d = 3
rounds = d
coupler_name = 'zzz'

p1 = UnrotatedSurfaceCode(distance=d)
p1.transpose_coords()

p2 = UnrotatedSurfaceCode(distance=d)
p2.transpose_coords()

p3 = UnrotatedSurfaceCode(distance=d)
p3.transpose_coords()

system = QECSystem()
system.add_patch(p1, name='p1')                                                  # (0,4, 0,4)
system.add_patch(p2, name='p2', offset=(10, 0))                                  # (10,14, 0,4)
system.add_patch(p3, name='p3', offset=(0, 6))                                   # (0,4, 6,10)
system.register_coupler(UnrotatedMultiPatchCoupler(), patch_names=['p1', 'p2', 'p3'],
                        name=coupler_name, path_axis='vertical', center_axis=7.0)

num_qubits = system.num_qubits
cp = system.coupler_patches[coupler_name]

tracker = SyndromeTracker(num_qubits=num_qubits, expected_num_logicals=system.num_logicals)
builder = CircuitBuilder(tracker=tracker, system_config=system, if_detector=True)
builder.write_coordinates()

init_dict = {q: 'Z' for q in system.data_indices if system.index_to_owner_map[q] != coupler_name}
builder.initialize(init_dict=init_dict, n=num_qubits)

se = UnrotatedSurfaceCodeExtractionBlock(system)
builder.apply_syndrome_extraction(circuit_chunk=se.circuit, rounds=rounds)

builder.activate_coupler(coupler_name)
coupler_data_global = [system.local_to_global_map[coupler_name][q] for q in cp.data_indices]
coupler_init = {q: 'X' for q in coupler_data_global}
builder.initialize(init_dict=coupler_init, n=num_qubits)

se2 = UnrotatedSurfaceCodeExtractionBlock(system)
builder.apply_syndrome_extraction(circuit_chunk=se2.circuit, rounds=rounds)

measure_dict = {q: 'Z' for q in system.data_indices if system.index_to_owner_map[q] != coupler_name}
measure_dict.update(coupler_init)
builder.apply_data_readout(final_measurements=measure_dict)

circuit = builder.circuit
print(f"Circuit: {circuit.num_qubits} qubits, {circuit.num_detectors} detectors, {circuit.num_observables} obs")
dem = circuit.detector_error_model(decompose_errors=True)
print(f"DEM OK: {dem.num_detectors} det, {dem.num_observables} obs")

circuit.diagram("detslice-with-ops-svg")